In [1]:
# Importing Libraries
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
#Connecting to the Database
host = "localhost"
port = '3306'
database = "text_to_sql"
username = "root"
password = "pass123"
mysqluri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"

db = SQLDatabase.from_uri(mysqluri,sample_rows_in_table_info=1)

In [4]:
db.get_table_info()

'\nCREATE TABLE `2017_budgets` (\n\t`Product Name` TEXT, \n\t`2017 Budgets` DOUBLE\n)COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB DEFAULT CHARSET=utf8mb4\n\n/*\n1 rows from 2017_budgets table:\nProduct Name\t2017 Budgets\nProduct 1\t3016489.2089999998\n*/\n\n\nCREATE TABLE customers (\n\t`Customer Index` INTEGER, \n\t`Customer Names` TEXT\n)COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB DEFAULT CHARSET=utf8mb4\n\n/*\n1 rows from customers table:\nCustomer Index\tCustomer Names\n1\tGeiss Company\n*/\n\n\nCREATE TABLE products (\n\t`Index` INTEGER, \n\t`Product Name` TEXT\n)COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB DEFAULT CHARSET=utf8mb4\n\n/*\n1 rows from products table:\nIndex\tProduct Name\n1\tProduct 1\n*/\n\n\nCREATE TABLE regions (\n\tid INTEGER, \n\tname TEXT, \n\tcounty TEXT, \n\tstate_code TEXT, \n\tstate TEXT, \n\ttype TEXT, \n\tlatitude DOUBLE, \n\tlongitude DOUBLE, \n\tarea_code INTEGER, \n\tpopulation INTEGER, \n\thouseholds INTEGER, \n\tmedian_income INTEGER, \n\tland_area INTEGER, \

In [5]:
# Create the LLM Prompt Template
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question. 
Remember : Only write the SQL query and nothing else. Do not include any explanations or comments. Provide me SQL query in single line without any line breaks.
Table Schema: {table_info}
Question: {question}
SQL Query:
"""
promt = ChatPromptTemplate.from_template(template)

In [13]:
#get the schema of the database
def get_schema(db_schema):
    return db_schema.get_table_info()

In [24]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                            api_key="API_key")#Replace with your API key)

In [21]:
 #Create the sql query chain using the LLM and the prompt template 

sql_chain = (RunnablePassthrough.assign(table_info = lambda _:get_schema(db)) 
            | promt 
            | llm.bind(stop=["\nSQLResult:"]) 
            | StrOutputParser()) 

In [23]:
#test the chain with a question
question = "What is the total 'Line Total' for Geiss Company?"
resp = sql_chain.invoke({"question":question})
print(resp)

SELECT SUM(T1.`Line Total`) FROM sales_order AS T1 INNER JOIN customers AS T2 ON T1.`Customer Name Index` = T2.`Customer Index` WHERE T2.`Customer Names` = 'Geiss Company'
